In [49]:
import random
import math

In [ ]:
class PerudoGame:

    def __init__(self, players = 2, dice_per_player = 5, debug_mode = False, max_dice = 5):
        
        self.players = []
        self.dice_per_player = dice_per_player
        
        for i in range(players):
            self.players.append(Player(f"Player {i}",dice_per_player))

        self.game_over = False
        self.max_dice = max_dice
        self.debug_mode = debug_mode
        self.previous_bids = []
        self.current_player = None
        self.current_bid = None
        self.challenger = None
        self.next_start_player = None
        self.is_palifico = False
        

#   UI and debug functions
    def get_dice_counts(self):
        dice_count = {}
        for player in self.players:
            dice_count[player.name] = len(player.dice)
        return dice_count
    
    def show_game_state(self, player):
        print(f"Selected Player: {self.current_player.name}")
        if self.debug_mode:
            for player_i in self.players:
                print(f"{player_i.name}'s Dice: {player_i.dice}")
        else:
            print(f"Selected player's Dice: {player_i.dice}")
        print(f"Number of dice: {self.get_dice_counts()}")

        if self.is_palifico:
            print("PALIFICO")
        if self.current_bid != None:
            print(f"Current Bid:{self.current_bid.quantity, self.current_bid.face}")




    def reset(self):
        
        for player in self.players:
            player.reset()
        self.previous_bids = []
    
        
        self.current_player = self.players[0]
        self.next_start_player = None
        self.current_bid = None
        self.challenger = None

        
     
#   game loop functions
    def next_round(self):
        self.remove_players_with_no_dice()
        if self.check_is_winner():
            return
        self.current_player = self.next_start_player
        self.previous_bids = []
        self.current_bid = None
        self.players_roll()
        self.is_palifico = False
    def players_roll(self):
        for player in self.players:
            player.roll()
    def check_palifico(self, action):
        if len(action.player.dice) == 1 and action.player.palifico == False:
            action.player.palifico = True
            self.is_palifico = True
    def perform_action(self, action):
            self.check_palifico(action)
            if action.action_type == "bid":
                if self.is_bid_legal(action):
                    self.apply_bid(action)
                    self.next_player()
                    return True
                else:
                    print("Invalid bid. Please try again.")
                    return False
                
            elif action.action_type == "call":
                
                self.resolve_call()
                self.next_round()
            elif action.action_type == "perudo":
                print(f"Perudo Called on {self.current_bid.quantity}, {self.current_bid.face}'s")
                self.resolve_perudo()
                self.next_round()
            else:
                raise ValueError(f"Unknown action type: {action.action_type}")
            return True
    def next_player(self):
        current_index = self.players.index(self.current_player)
        self.current_player = self.players[(current_index + 1) % len(self.players)]
        #self.current_player.make_action
    def remove_players_with_no_dice(self):
        for player in self.players.copy():
            if len(player.dice) == 0:
                self.players.remove(player)
    def check_is_winner(self):
        if len(self.players) == 1:
            print(f"{self.players[0].name} has won")
            self.game_over = True
            return True
        return False

#   bid creation and testing functions   
    def is_bid_legal(self, action):
        
        if 1> action.face > 6 or action.quantity < 1:
            return False
        elif self.current_bid is None:
            return True
        elif self.is_palifico:
            return self.is_bid_legal_palifico(action)
        elif self.current_bid.face == 1:
            return self.is_leaving_ones_bid_legal(action)
        elif action.face == 1:
            return self.is_switch_to_ones_legal(action)
        return self.is_normal_bid_legal(action)

    def is_bid_legal_palifico(self, action):
        return action.face == self.current_bid.face and action.quantity > self.current_bid.quantity
            
    def is_leaving_ones_bid_legal(self, action):
        if action.face == 1:
            return action.quantity > self.current_bid.quantity
        return action.quantity >= self.current_bid.quantity * 2 and self.check_previous_bids(action)
    def is_normal_bid_legal(self, action):
        return (
            action.quantity > self.current_bid.quantity 
            or action.face > self.current_bid.face
            )
    def is_switch_to_ones_legal(self, action):
        return (
            self.check_previous_bids(action) 
            and (action.quantity == (self.current_bid.quantity+1)//2 
            and action.face == 1)
              )

    def get_previous_non_one_bid(self):
        for bid in reversed(self.previous_bids):
            if bid.face != 1:
                return bid
    
        
    def check_previous_bids(self, action):
        if action.face == 1:
            for previous_action in self.previous_bids:
                if previous_action.quantity == action.quantity and previous_action.face == action.face:
                    return False
        if action.face != 1:
            previous_bid = self.get_previous_non_one_bid()
            if action.quantity>previous_bid.quantity or action.face > previous_bid.face:
                return True
        return True
    
    def apply_bid(self, action):
        self.current_bid = action
        self.previous_bids.append(action)
        pass

#   calling bluff functions
    def count_face(self, face):
        count = 0
        for player in self.players:
            count += player.dice.count(face)
            # 1s are jokers unless el palifico rules apply
            if face != 1 and self.is_palifico == False:
                count += player.dice.count(1)
        return count
    def is_call_false(self):
        return self.count_face(self.current_bid.face)>=self.current_bid.quantity
    def resolve_call(self):
        challenger = self.current_player
        if self.is_call_false():
            self.next_start_player = challenger
            challenger.remove_die()
        else:
            self.next_start_player = self.current_bid.player
            self.current_bid.player.remove_die()
#   perudo function
    def resolve_perudo(self):
        self.next_start_player = self.current_player
        if self.count_face(self.current_bid.face) == self.current_bid.quantity:
            if len(self.current_player.dice) < self.max_dice:
                self.current_player.add_die()
            print(f"{self.current_player.name}'s perudo was succesful")
        else:
            self.current_player.remove_die()
            print(f"{self.current_player.name}'s perudo was unsuccesful")


    

In [51]:
class Player:
    
    def __init__(self, name: str, initial_dice_quantity):
        
        self.name = name
        self.initial_dice_quantity = initial_dice_quantity
        self.dice = []
        self.palifico = False
        
    def reset(self):
            self.dice = []
            for i in range(self.initial_dice_quantity):
                        self.dice.append(random.randint(1,6))


    # control functions
    def remove_die(self):
        self.dice.pop()
    def add_die(self):
        self.dice.append(1)
    def roll(self):
        for i in range(len(self.dice)):
            self.dice[i] = random.randint(1,6)
    
    
    # turn actions
    def make_bid(self, quantity, face):
        return Action("bid", self, quantity, face)
    def call_bluff(self):
        return Action("call", self)
    def call_perudo(self):
        return Action("perudo", self)

In [52]:
class Action:
    def __init__(self, action_type, player, quantity = None, face = None):
        self.action_type = action_type
        self.player = player
        self.quantity = quantity
        self.face = face

In [ ]:
game = PerudoGame(2, 2, debug_mode=True)
game.reset()

while not game.game_over:
    
    game.show_game_state(game.current_player)

    action_type = input("Action (bid/call/perudo): ")

    if action_type == "bid":

        quantity, face = map(int, input("Quantity Face: ").split())
        

        action = Action(
            "bid",
            game.current_player,
            quantity,
            face
        )

    elif action_type == "call":

        action = Action(
            "call",
            game.current_player
        )

    elif action_type == "perudo":

        action = Action(
            "perudo",
            game.current_player
        )
    elif action_type == "break":
        break
    else:
        print("Invalid action.")
        continue

    game.perform_action(action)

Selected Player: Player 0
Player 0's Dice: [5, 3]
Player 1's Dice: [1, 1]
Number of dice: {'Player 0': 2, 'Player 1': 2}
Selected Player: Player 1
Player 0's Dice: [5, 3]
Player 1's Dice: [1, 1]
Number of dice: {'Player 0': 2, 'Player 1': 2}
Current Bid:(3, 5)
Perudo Called on 3, 5's
Player 1's perudo was succesful
Selected Player: Player 1
Player 0's Dice: [5, 3]
Player 1's Dice: [1, 5, 4]
Number of dice: {'Player 0': 2, 'Player 1': 3}
Selected Player: Player 0
Player 0's Dice: [5, 3]
Player 1's Dice: [1, 5, 4]
Number of dice: {'Player 0': 2, 'Player 1': 3}
Current Bid:(2, 2)
Selected Player: Player 1
Player 0's Dice: [6, 4]
Player 1's Dice: [2, 3]
Number of dice: {'Player 0': 2, 'Player 1': 2}
Selected Player: Player 0
Player 0's Dice: [6, 4]
Player 1's Dice: [2, 3]
Number of dice: {'Player 0': 2, 'Player 1': 2}
Current Bid:(1, 1)
Selected Player: Player 1
Player 0's Dice: [3, 4]
Player 1's Dice: [2]
Number of dice: {'Player 0': 2, 'Player 1': 1}
Selected Player: Player 0
Player 0's 